In [ ]:
# ── Cell 1: Setup — run once per session ──────────────────────────────────
!pip install -q timm einops ml-collections medpy SimpleITK tensorboardX thop

!cp -r /kaggle/input/datasets/deepsotaai/adada-transunet-code/Ada-DA-TransUNet /kaggle/working/Ada-DA-TransUNet
!cp -r /kaggle/input/datasets/deepsotaai/vit-pretrained-weights/model            /kaggle/working/model

# IMPORTANT: clear stale __pycache__ compiled bytecode from the uploaded dataset.
# Without this, Python silently runs old .pyc files even after you patch .py files.
import shutil, os
for root, dirs, files in os.walk('/kaggle/working/Ada-DA-TransUNet'):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))
            print(f'Cleared: {os.path.join(root, d)}')

# Kaggle strips '+' from filenames — rename back
!mv /kaggle/working/model/vit_checkpoint/imagenet21k/R50ViT-B_16.npz \
    /kaggle/working/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz 2>/dev/null || true

# Prevent HuggingFace 'datasets' library from shadowing local datasets/ folder
!touch /kaggle/working/Ada-DA-TransUNet/datasets/__init__.py

# Symlink Synapse data
!mkdir -p /kaggle/working/data/Synapse
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz  /kaggle/working/data/Synapse/train_npz
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5 /kaggle/working/data/Synapse/test_vol_h5

print('Setup complete.')

In [ ]:
# ── Cell 2: Verify all DataParallel .contiguous() fixes ───────────────────
# Reads source files directly (not __pycache__) so the check is always honest.
# Any MISSING check is auto-patched in-place so training can proceed.

base = '/kaggle/working/Ada-DA-TransUNet/Architecture'

with open(f'{base}/block.py') as f:
    src_block = f.read()
with open(f'{base}/AdaDATransUNet.py') as f:
    src_arch = f.read()

checks = {
    'window_partition':       'x = x.contiguous()' in src_block,
    'window_reverse':         'windows = windows.contiguous()' in src_block,
    'LowRankWindowedPAM':     'x_w.contiguous()' in src_block,
    'GroupedCAM_bmm1':        'x_g.transpose(1, 2).contiguous()' in src_block,
    'GroupedCAM_bmm2':        'X.transpose(1, 2).contiguous()' in src_block,
    'DecoderBlock_skip':      'skip = skip.contiguous()' in src_arch,
    'ViT_embedding':          'x.transpose(-1, -2).contiguous()' in src_arch,
    'Attention_permute':      'permute(0, 2, 1, 3).contiguous()' in src_arch,
    'Attention_QK_matmul':    'key_layer.transpose(-1, -2).contiguous()' in src_arch,
    'Attention_hidden_cont':  'hidden_states = hidden_states.contiguous()' in src_arch,
    'cublaslt_disabled':      'preferred_blas_library' in open('/kaggle/working/Ada-DA-TransUNet/train.py').read(),
}

for k, v in checks.items():
    print(f"{'OK    ' if v else 'MISSING'} {k}")

missing = [k for k, v in checks.items() if not v]
if not missing:
    print('\nAll checks PASSED — safe to run training.')
else:
    print(f'\n{len(missing)} check(s) MISSING — applying patches...')

    # --- block.py patches ---
    patched_block = False

    if 'window_partition' in missing:
        src_block = src_block.replace(
            'def window_partition(x, window_size):\n    B, C, H, W = x.shape',
            'def window_partition(x, window_size):\n    x = x.contiguous()  # DataParallel scatter produces non-contiguous views\n    B, C, H, W = x.shape'
        )
        patched_block = True
        print('  Patched: window_partition')

    if 'window_reverse' in missing:
        src_block = src_block.replace(
            'def window_reverse(windows, window_size, H, W):\n    M = window_size',
            'def window_reverse(windows, window_size, H, W):\n    windows = windows.contiguous()\n    M = window_size'
        )
        patched_block = True
        print('  Patched: window_reverse')

    if 'LowRankWindowedPAM' in missing:
        src_block = src_block.replace(
            'x_n = x_w.view(nBW, C, M * M)',
            'x_n = x_w.contiguous().view(nBW, C, M * M)'
        )
        patched_block = True
        print('  Patched: LowRankWindowedPAM')

    if 'GroupedCAM_bmm1' in missing:
        src_block = src_block.replace(
            'X = torch.bmm(x_g, x_g.transpose(1, 2))',
            'X = torch.bmm(x_g, x_g.transpose(1, 2).contiguous())'
        )
        patched_block = True
        print('  Patched: GroupedCAM_bmm1')

    if 'GroupedCAM_bmm2' in missing:
        src_block = src_block.replace(
            'E_g = torch.bmm(X.transpose(1, 2), x_g)',
            'E_g = torch.bmm(X.transpose(1, 2).contiguous(), x_g)'
        )
        patched_block = True
        print('  Patched: GroupedCAM_bmm2')

    if patched_block:
        with open(f'{base}/block.py', 'w') as f:
            f.write(src_block)

    # --- AdaDATransUNet.py patches ---
    patched_arch = False

    if 'DecoderBlock_skip' in missing:
        src_arch = src_arch.replace(
            'def forward(self, x, skip=None):\n        x = nn.UpsamplingBilinear2d(scale_factor=2)(x)',
            'def forward(self, x, skip=None):\n        x = nn.UpsamplingBilinear2d(scale_factor=2)(x)\n        if skip is not None:\n            skip = skip.contiguous()'
        )
        patched_arch = True
        print('  Patched: DecoderBlock_skip')

    if 'ViT_embedding' in missing:
        src_arch = src_arch.replace(
            'x = x.transpose(-1, -2)',
            'x = x.transpose(-1, -2).contiguous()'
        )
        patched_arch = True
        print('  Patched: ViT_embedding')

    if 'Attention_permute' in missing:
        src_arch = src_arch.replace(
            'return x.permute(0, 2, 1, 3)',
            'return x.permute(0, 2, 1, 3).contiguous()'
        )
        patched_arch = True
        print('  Patched: Attention_permute')

    if 'Attention_QK_matmul' in missing:
        src_arch = src_arch.replace(
            'attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))',
            'attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2).contiguous())'
        )
        patched_arch = True
        print('  Patched: Attention_QK_matmul')

    if 'Attention_hidden_cont' in missing:
        src_arch = src_arch.replace(
            '    def forward(self, hidden_states):\n        mixed_query_layer = self.query(hidden_states)',
            '    def forward(self, hidden_states):\n        hidden_states = hidden_states.contiguous()  # ensure aligned ptr for cublasLt on replica 1\n        mixed_query_layer = self.query(hidden_states)'
        )
        patched_arch = True
        print('  Patched: Attention_hidden_cont')

    if patched_arch:
        with open(f'{base}/AdaDATransUNet.py', 'w') as f:
            f.write(src_arch)

    # --- train.py patches ---
    if 'cublaslt_disabled' in missing:
        train_path = '/kaggle/working/Ada-DA-TransUNet/train.py'
        with open(train_path) as f:
            src_train = f.read()
        src_train = src_train.replace(
            'import torch\nimport torch.backends.cudnn as cudnn',
            'import torch\nimport torch.backends.cudnn as cudnn\n\n# Disable cublasLt fused gemm — prevents CUBLAS_STATUS_EXECUTION_FAILED on Kaggle T4\ntorch.set_float32_matmul_precision(\'highest\')\ntry:\n    torch.backends.cuda.preferred_blas_library("cublas")\nexcept AttributeError:\n    pass'
        )
        with open(train_path, 'w') as f:
            f.write(src_train)
        print('  Patched: cublaslt_disabled in train.py')

    print('\nAll patches applied. DO NOT re-run setup — it will overwrite.')
    print('Proceed to training cell.')

In [ ]:
%%bash
# ── Cell 3: Training ──────────────────────────────────────────────────────
# Tip: if you still hit cudaErrorMisalignedAddress, add the line below to
# get the EXACT error location (makes training synchronous/slow, debug only):
#   export CUDA_LAUNCH_BLOCKING=1

echo "========================================" 
echo " AdaDA-TransUNet  |  TRAINING"
echo " Started: $(date)"
echo "========================================" 
cd /kaggle/working/Ada-DA-TransUNet
python -u train.py \
  --dataset      Synapse \
  --vit_name     R50-ViT-B_16 \
  --max_epochs   300 \
  --batch_size   24 \
  --base_lr      0.01 \
  --n_gpu        2 \
  --n_skip       3 \
  --img_size     224 \
  --window_size  7 \
  --rank         32 \
  --groups       8 \
  --seed         1234 \
  --val_interval 10
echo "========================================" 
echo " Training finished: $(date)"
echo "========================================" 

In [ ]:
%%bash
# ── Cell 4: Inference ─────────────────────────────────────────────────────
echo "========================================" 
echo " AdaDA-TransUNet  |  INFERENCE"
echo " Started: $(date)"
echo "========================================" 
cd /kaggle/working/Ada-DA-TransUNet
python -u test.py \
  --dataset      Synapse \
  --vit_name     R50-ViT-B_16 \
  --num_classes  9 \
  --img_size     224 \
  --window_size  7 \
  --rank         32 \
  --groups       8 \
  --is_savenii
echo "========================================" 
echo " Inference finished: $(date)"
echo "========================================" 